# Distil BioCLIP-2 into a deployable ViT-B

Closes the 19pp coverage gap between the encoder that scores (BioCLIP-2, a 304M
ViT-L that cannot ship) and the one that fits a phone (BioCLIP v1, 46 MB at int4).

The teacher never runs here — its 82k embeddings are already cached — so this is a
regression onto vectors, not a two-model training job.

**Runtime: A100. Set Runtime → Change runtime type → A100 before running.**


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv


## 1. Code and data

Two things must be uploaded or mounted: the repo, and `data/processed/` containing
the cached teacher embeddings (`catalog_*_bioclip2.npz`, `background_*_bioclip2.npz`)
plus the images they refer to (`images/`, `images_background/`). That is ~6 GB —
Drive is usually less painful than re-uploading.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# adjust to wherever the repo and data live
REPO = '/content/drive/MyDrive/plantid'
%cd $REPO


In [ ]:
!pip -q install open_clip_torch


## 2. Check the transfer set before training anything

This is the correctness check that matters. The student must never see an image
we evaluate on: not the catalogue test split, not the catalogue val split (used
for temperature scaling), and none of the iNaturalist observations. Expect
**~48,564 images and `any iNat: False`**.


In [ ]:
import sys; sys.path.insert(0, '.')
from plantid.train.distil import build_transfer_set

df = build_transfer_set()
print(f'transfer set: {len(df):,} images, teacher dim {df.teacher.iloc[0].shape[0]}')
print('catalogue :', sum(p.startswith('images/') for p in df.local_path))
print('background:', sum(p.startswith('images_background/') for p in df.local_path))
print('any iNat  :', any('images_inat' in p for p in df.local_path))


## 3. Train

~48k images at ~700 img/s on an A100 is roughly a minute per epoch, so 40 epochs
is under an hour. Watch the two cosines: if **train keeps rising while held-out
flattens**, the student is memorising the transfer set and the fix is more images
(PlantNet has ~244k not yet downloaded), not more epochs.


In [ ]:
!python -m plantid.train.distil \
    --epochs 40 --batch-size 256 --lr 1e-4 --head-lr 1e-3 \
    --workers 8 --out data/processed/distil/student.pt


In [ ]:
import json, matplotlib.pyplot as plt
h = json.load(open('data/processed/distil/student.json'))
ep = [r['epoch'] for r in h['history']]
plt.plot(ep, [r['train_cos'] for r in h['history']], label='train')
plt.plot(ep, [r['val_cos'] for r in h['history']], label='held out')
plt.axhline(h['baseline_cos'], ls='--', c='grey', label='at init')
plt.xlabel('epoch'); plt.ylabel('cosine to teacher'); plt.legend(); plt.show()


## 4. Score it the same way as every other encoder

`bioclip1_distil` is registered in `ENCODERS`, so the existing pipeline takes it
unchanged. Embedding all three corpora is ~11 minutes on an A100.


In [ ]:
from plantid.features import embed_catalog, embed_background, embed_inat
for mod in (embed_catalog, embed_background, embed_inat):
    mod.main(variant='bioclip1_distil')


In [ ]:
!python -m plantid.eval.rejection --variant bioclip1_distil


## 5. The bar it has to clear

Against BioCLIP v1, on the held-out iNaturalist set:

| | genus | 95% CI | species | precision @20% | coverage |
|---|---|---|---|---|---|
| BioCLIP-2 (teacher, cannot ship) | 0.9747 | [0.9653, 0.9827] | 0.8460 | 0.956 | 0.722 |
| BioCLIP v1 (what ships today) | 0.9310 | [0.9163, 0.9440] | 0.7604 | 0.946 | 0.531 |
| **distilled student** | ? | | | | |

**Pass = beats BioCLIP v1 by a margin whose paired interval excludes zero.**
Anything less and the honest conclusion is that feature distillation on 48k
images does not close this gap — in which case the remaining options are raising
the size budget for BioCLIP-2 (~164 MB at int4) or accepting ~50% coverage.

If it does pass, the deployment path is already measured: Core ML int4 export
costs ~1.3pp of genus accuracy, and `computeUnits` must be pinned to
`.cpuAndNeuralEngine` (see `ONDEVICE_FINDINGS.md`).


In [ ]:
import sys; sys.path.insert(0, '.')
import numpy as np
from plantid.config import DATA_PROCESSED as D
from plantid.eval.rejection import build_observations, cluster_bootstrap

F = {v: build_observations(str(D/f'inat_{v}.npz'), variant=v)[0]
     for v in ('bioclip1', 'bioclip1_distil')}
m  = F['bioclip1'].in_catalog.values
sp = F['bioclip1'].species.values[m]
for col in ('genus_ok', 'species_ok'):
    d = F['bioclip1_distil'][col].values[m].astype(float) - F['bioclip1'][col].values[m].astype(float)
    lo, hi = cluster_bootstrap(d, sp)
    star = '' if lo <= 0 <= hi else '  * excludes zero'
    print(f'{col:11s} distilled - bioclip1: {d.mean():+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]{star}')
